<a href="https://colab.research.google.com/github/orionrjsun-collab/BUS4-118s/blob/main/ReACTCodeGeneration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

You are an expert Python programming assistant operating in a ReACT loop (Reason → Act → Observe → Reflect → Repeat). You will solve problems by explicitly cycling through these labeled stages. Never skip a stage. Always label each stage clearly in your response.

Stages you must always use:
[REASON] — Analyze the problem. State your plan and what you expect.
[ACT] — Write complete, runnable Python code in a single code block.
[OBSERVE] — Show the actual output or describe what would happen when the code runs.
[REFLECT] — Evaluate correctness. Identify any bugs, edge cases, or improvements needed.
[REPEAT or DONE] — If issues were found, loop again from [REASON]. If the solution is correct, write DONE and summarize.

Using the ReACT loop format (REASON → ACT → OBSERVE → REFLECT → REPEAT/DONE), write and execute Python code that solves the following task:

TASK:
Read a CSV file containing student names and their scores, then compute and print:
1. The mean, median, and standard deviation of all scores.
2. The name and score of the highest-scoring student.
3. A letter grade for each student (A: 90+, B: 80–89, C: 70–79, D: 60–69, F: below 60).

CONSTRAINTS:
- Use only these libraries: pandas, numpy, csv (stdlib), statistics (stdlib).
- Do NOT use scikit-learn or scipy.
- Input: a CSV with exactly two columns — "name" (string) and "score" (integer 0–100).
- If the file is missing or malformed, catch the exception and print a descriptive error message; do not crash.
- If a score is outside 0–100, flag that row as invalid and skip it in calculations.
- Output must be clearly labeled (e.g., "Mean Score: 84.3").
- Since we are running in Colab without a real file, programmatically create the CSV first using Python before reading it.
- Include sample data with at least 8 students, including one score outside 0–100 to test edge case handling.

FORMAT REQUIREMENTS:
- Each stage must be labeled: [REASON], [ACT], [OBSERVE], [REFLECT], [REPEAT] or [DONE].
- The [ACT] stage must contain one complete, self-contained, copy-pasteable Python code block.
- The [OBSERVE] stage must show the exact printed output (you may simulate it if you cannot execute, but clearly state if it is simulated vs. real).
- Complete at least 2 full iterations of the ReACT loop — first attempt, then one refinement pass.

Now verify your solution meets all requirements. Go through this checklist and respond YES or NO for each item:

1. Did the response include clearly labeled [REASON], [ACT], [OBSERVE], [REFLECT], and [DONE] stages?
2. Does the code create the CSV programmatically before reading it?
3. Does the code compute mean, median, and standard deviation?
4. Does the code identify the highest-scoring student by name?
5. Does the code assign letter grades to every valid student?
6. Does the code handle a missing or malformed file with a try/except block?
7. Does the code detect and skip scores outside 0–100, with a printed warning?
8. Is the output clearly labeled?
9. Did the loop run at least 2 full iterations?
10. Is the final code block self-contained and runnable in Google Colab with no modifications?

If any answer is NO, re-enter the ReACT loop from [REASON] and fix the issue. Then show the corrected final code block labeled: FINAL SOLUTION.

In [1]:
import pandas as pd
import numpy as np  # allowed (not strictly needed now, but kept within constraints)
import csv
import statistics
import tempfile
import os
import math

def main():
    # 1) Create a sample CSV file
    rows = [
        ("Alice", 92),
        ("Bob", 85),
        ("Carla", 78),
        ("Diego", 66),
        ("Eve", 59),
        ("Fay", 100),
        ("Gus", 73),
        ("Hana", 88),
        ("Ivan", 105),  # invalid score (outside 0–100)
    ]

    tmpdir = tempfile.mkdtemp()
    csv_path = os.path.join(tmpdir, "students.csv")

    with open(csv_path, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["name", "score"])
        w.writerows(rows)

    # 2) Read and process with stronger checks
    try:
        df = pd.read_csv(csv_path, dtype={"name": "string"})

        expected = {"name", "score"}
        if set(df.columns) != expected or len(df.columns) != 2:
            raise ValueError(f"CSV must have exactly two columns {sorted(expected)}; got {list(df.columns)}")

        # Enforce column order for consistent downstream printing
        df = df[["name", "score"]]

        # Coerce score to numeric
        df["score"] = pd.to_numeric(df["score"], errors="coerce")

        # Flag invalid rows (NaN or outside 0–100)
        invalid_mask = df["score"].isna() | (df["score"] < 0) | (df["score"] > 100)
        invalid_rows = df[invalid_mask]
        valid_df = df[~invalid_mask].copy()

        if not invalid_rows.empty:
            print("Invalid rows (skipped in calculations):")
            for i, (idx, r) in enumerate(invalid_rows.iterrows(), start=1):
                print(f" {i}. Row {idx}: name={r['name']}, score={r['score']}")
        else:
            print("Invalid rows (skipped in calculations): none")

        scores = valid_df["score"].astype(int).tolist()
        if not scores:
            print("Error: No valid scores found (after removing invalid rows).")
            return

        # Stats (explicitly sample std dev)
        mean = statistics.mean(scores)
        median = statistics.median(scores)

        if len(scores) >= 2:
            stdev = statistics.stdev(scores)  # sample stdev
            stdev_line = f"Standard Deviation (sample): {stdev:.2f}"
        else:
            stdev_line = "Standard Deviation (sample): N/A (need at least 2 valid scores)"

        print(f"Mean Score: {mean:.2f}")
        print(f"Median Score: {median:.2f}")
        print(stdev_line)

        # Top student among valid
        top_idx = valid_df["score"].idxmax()
        top_student = str(valid_df.loc[top_idx, "name"])
        top_score = int(valid_df.loc[top_idx, "score"])
        print(f"Top Student: {top_student} ({top_score})")

        # Grades (invalid rows labeled)
        def letter(s: int) -> str:
            return ("A" if s >= 90 else
                    "B" if s >= 80 else
                    "C" if s >= 70 else
                    "D" if s >= 60 else
                    "F")

        print("Grades:")
        for _, r in df.iterrows():
            name = str(r["name"])
            sc = r["score"]
            if pd.isna(sc) or sc < 0 or sc > 100:
                print(f" - {name}: {sc} -> INVALID")
            else:
                s_int = int(sc)
                print(f" - {name}: {s_int} -> {letter(s_int)}")

    except FileNotFoundError:
        print(f"Error: File not found at path: {csv_path}")
    except pd.errors.EmptyDataError:
        print("Error: CSV file is empty or unreadable.")
    except Exception as e:
        print(f"Error reading or processing CSV: {e}")

if __name__ == "__main__":
    main()

Invalid rows (skipped in calculations):
 1. Row 8: name=Ivan, score=105
Mean Score: 80.12
Median Score: 81.50
Standard Deviation (sample): 13.75
Top Student: Fay (100)
Grades:
 - Alice: 92 -> A
 - Bob: 85 -> B
 - Carla: 78 -> C
 - Diego: 66 -> D
 - Eve: 59 -> F
 - Fay: 100 -> A
 - Gus: 73 -> C
 - Hana: 88 -> B
 - Ivan: 105 -> INVALID
